In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
from peft import PeftModel

import torch
import json
import re
import os

from datetime import datetime
from collections import Counter

In [7]:
import os
from huggingface_hub import login

# Accept the gated dataset at huggingface.co/datasets/abhishek9909/train-time-opt-2 first.
login(token=os.environ["HF_TOKEN"]) if os.environ.get("HF_TOKEN") else login()

In [10]:
from pathlib import Path

DRIVE_MODELS_ROOT = Path("/content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models")

print("DRIVE_MODELS_ROOT:", DRIVE_MODELS_ROOT)
print("exists:", DRIVE_MODELS_ROOT.exists())
print("is_dir:", DRIVE_MODELS_ROOT.is_dir())

for p in sorted(DRIVE_MODELS_ROOT.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"{kind}  {p.name}")

DRIVE_MODELS_ROOT: /content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models
exists: True
is_dir: True
DIR   condition_0_llama3.1-8b_seed123_beta0p0
DIR   condition_0_qwen2.5-1.5b_seed7_beta0p0
DIR   condition_0_qwen2.5-3b_seed42_beta0p0
DIR   condition_0_qwen2.5-3b_seed7_beta0p0
DIR   condition_0_qwen2.5-7b_seed7_beta0p0
DIR   condition_0_unbiased_qwen2.5-1.5b_seed42_beta0p0
DIR   condition_0_unbiased_qwen2.5-3b_seed42_beta0p0
DIR   condition_recovery_qwen2.5-1.5b_seed7_beta0p0
DIR   condition_recovery_qwen2.5-3b_seed42_beta0p0
FILE  mmlu_eval_log_qwen_3b_balanced.jsonl
FILE  mmlu_test_10_per_subject_balanced.jsonl


In [11]:
import re
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

MAX_SEQ_LEN = 1024
CHECKPOINT_STEP = "checkpoint-200"
EVAL_BATCH_SIZE = 8
SKIP_IF_EXISTS = True  # skip runs that already have mmlu_eval_log.jsonl on Drive

# Gemma MMLU targets (24 runs with completed adapters on HF):
#   - biased condition_0: gemma3-1b/4b/12b/27b × seeds 7, 42, 123
#   - unbiased + recovery: gemma3-1b/4b × seeds 7, 42, 123
# Set to None to eval every condition_* folder on HF.
GEMMA_MODELS_ALL = ("gemma3-1b", "gemma3-4b", "gemma3-12b", "gemma3-27b")
GEMMA_MODELS_1B_4B = ("gemma3-1b", "gemma3-4b")
GEMMA_SEEDS = (7, 42, 123)


def gemma_mmlu_target_dirs():
    dirs = []
    for model in GEMMA_MODELS_ALL:
        for seed in GEMMA_SEEDS:
            dirs.append(f"condition_0_{model}_seed{seed}_beta0p0")
    for model in GEMMA_MODELS_1B_4B:
        for seed in GEMMA_SEEDS:
            dirs.append(f"condition_0_unbiased_{model}_seed{seed}_beta0p0")
            dirs.append(f"condition_recovery_{model}_seed{seed}_beta0p0")
    return dirs


MMLU_TARGET_DIR_NAMES = gemma_mmlu_target_dirs()

# Results + MMLU questions on Drive
# https://drive.google.com/drive/folders/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_
DRIVE_MODELS_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models"
)
MMLU_QUESTIONS_PATH = DRIVE_MODELS_ROOT / "mmlu_test_10_per_subject_balanced.jsonl"

# LoRA adapters from gated HF dataset (fetched on demand during eval)
HF_DATASET_REPO = "abhishek9909/train-time-opt-2"
ADAPTER_CACHE = Path("/content/hf_adapter_cache")

BASE_MODEL_BY_SLUG = {
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen2.5-14b": "Qwen/Qwen2.5-14B-Instruct",
    "llama3.2-1b": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "llama3.2-3b": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "llama3.1-8b": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "gemma3-1b": "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "gemma3-4b": "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "gemma3-12b": "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "gemma3-27b": "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
}

BATCH_SIZE_BY_SLUG = {
    "qwen2.5-7b": 4,
    "qwen2.5-14b": 2,
    "llama3.1-8b": 4,
    "gemma3-12b": 2,
    "gemma3-27b": 1,
}

MODEL_SIZE_ORDER = {
    "qwen2.5-0.5b": 0,
    "llama3.2-1b": 1,
    "gemma3-1b": 2,
    "qwen2.5-1.5b": 3,
    "llama3.2-3b": 4,
    "gemma3-4b": 5,
    "qwen2.5-3b": 6,
    "qwen2.5-7b": 7,
    "llama3.1-8b": 8,
    "gemma3-12b": 9,
    "qwen2.5-14b": 10,
    "gemma3-27b": 11,
}

CONDITION_ORDER = {"biased": 0, "unbiased": 1, "recovered": 2}


def parse_run_dir(name: str):
    m = re.match(r"^condition_recovery_(?P<slug>.+)_seed(?P<seed>\d+)_beta0p0$", name)
    if m:
        return {
            "dir_name": name,
            "condition": "recovered",
            "model_slug": m.group("slug"),
            "seed": int(m.group("seed")),
        }
    m = re.match(r"^condition_0_unbiased_(?P<slug>.+)_seed(?P<seed>\d+)_beta0p0$", name)
    if m:
        return {
            "dir_name": name,
            "condition": "unbiased",
            "model_slug": m.group("slug"),
            "seed": int(m.group("seed")),
        }
    m = re.match(r"^condition_0_(?P<slug>.+)_seed(?P<seed>\d+)_beta0p0$", name)
    if m:
        return {
            "dir_name": name,
            "condition": "biased",
            "model_slug": m.group("slug"),
            "seed": int(m.group("seed")),
        }
    return None


def hf_adapter_subpath(run_meta):
    stage = (
        "outputs_stage2_recovery"
        if run_meta["condition"] == "recovered"
        else "outputs_stage2_reasoning_first"
    )
    return f"{run_meta['dir_name']}/{stage}/{CHECKPOINT_STEP}"


def discover_hf_runs():
    api = HfApi()
    tree_items = list(
        api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset", recursive=False)
    )

    # Don't use isinstance(RepoFolder): Colab can return a different class object
    # with the same repr. Top-level condition_* entries are always run folders.
    dir_names = sorted(
        item.path
        for item in tree_items
        if item.path.startswith("condition_")
        and not item.path.endswith(".json")
        and "/" not in item.path
    )

    if not dir_names:
        print(
            "WARNING: no condition_* folders found. "
            f"Sample repo entries: {tree_items[:5]}"
        )

    runs = []
    for name in dir_names:
        meta = parse_run_dir(name)
        if meta is None:
            continue
        base_model = BASE_MODEL_BY_SLUG.get(meta["model_slug"])
        if base_model is None:
            print(f"SKIP unknown model slug: {name}")
            continue

        results_dir = DRIVE_MODELS_ROOT / meta["dir_name"]
        results_dir.mkdir(parents=True, exist_ok=True)
        meta = {
            **meta,
            "hf_subpath": hf_adapter_subpath(meta),
            "base_model": base_model,
            "batch_size": BATCH_SIZE_BY_SLUG.get(meta["model_slug"], EVAL_BATCH_SIZE),
            "results_dir": results_dir,
            "eval_log_path": results_dir / "mmlu_eval_log.jsonl",
            "eval_summary_path": results_dir / "mmlu_eval_summary.json",
        }
        runs.append(meta)

    runs.sort(
        key=lambda r: (
            MODEL_SIZE_ORDER.get(r["model_slug"], 99),
            CONDITION_ORDER.get(r["condition"], 99),
            r["seed"],
        )
    )
    return runs


def adapter_exists_on_hf(run_meta) -> bool:
    api = HfApi()
    subpath = run_meta["hf_subpath"]
    for fname in ("adapter_config.json", "adapter_model.safetensors"):
        if not api.file_exists(
            HF_DATASET_REPO,
            f"{subpath}/{fname}",
            repo_type="dataset",
        ):
            return False
    return True


def download_adapter(run_meta) -> Path:
    """Download only the two files needed for inference."""
    subpath = run_meta["hf_subpath"]
    adapter_path = ADAPTER_CACHE / subpath
    needed = ("adapter_config.json", "adapter_model.safetensors")
    if all((adapter_path / fname).exists() for fname in needed):
        return adapter_path

    print(f"Downloading adapter: {subpath}")
    for fname in needed:
        hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename=f"{subpath}/{fname}",
            local_dir=ADAPTER_CACHE,
        )
    return adapter_path


def filter_target_runs(runs):
    if not MMLU_TARGET_DIR_NAMES:
        return runs
    wanted = set(MMLU_TARGET_DIR_NAMES)
    filtered = [r for r in runs if r["dir_name"] in wanted]
    missing = sorted(wanted - {r["dir_name"] for r in filtered})
    if missing:
        print("WARNING: target dirs not found on HF:", missing)
    return filtered


assert MMLU_QUESTIONS_PATH.exists(), f"MMLU file not found: {MMLU_QUESTIONS_PATH}"

ALL_HF_RUNS = filter_target_runs(discover_hf_runs())
print(f"MMLU questions: {MMLU_QUESTIONS_PATH}")
print(f"HF dataset:     {HF_DATASET_REPO}")
if MMLU_TARGET_DIR_NAMES:
    print(f"Target filter:  {len(MMLU_TARGET_DIR_NAMES)} remaining runs")
print(f"Found {len(ALL_HF_RUNS)} run folders to eval:\n")
for run in ALL_HF_RUNS:
    print(
        f"  {run['condition']:9s} seed={run['seed']:3d}  "
        f"{run['model_slug']:12s}  -> {run['dir_name']}"
    )

MMLU questions: /content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models/mmlu_test_10_per_subject_balanced.jsonl
Found 9 adapter runs on Drive:

  biased    seed=123  llama3.1-8b   -> condition_0_llama3.1-8b_seed123_beta0p0
  biased    seed=  7  qwen2.5-1.5b  -> condition_0_qwen2.5-1.5b_seed7_beta0p0
  biased    seed= 42  qwen2.5-3b    -> condition_0_qwen2.5-3b_seed42_beta0p0
  biased    seed=  7  qwen2.5-3b    -> condition_0_qwen2.5-3b_seed7_beta0p0
  biased    seed=  7  qwen2.5-7b    -> condition_0_qwen2.5-7b_seed7_beta0p0
  unbiased  seed= 42  qwen2.5-1.5b  -> condition_0_unbiased_qwen2.5-1.5b_seed42_beta0p0
  unbiased  seed= 42  qwen2.5-3b    -> condition_0_unbiased_qwen2.5-3b_seed42_beta0p0
  recovered seed=  7  qwen2.5-1.5b  -> condition_recovery_qwen2.5-1.5b_seed7_beta0p0
  recovered seed= 42  qwen2.5-3b    -> condition_recovery_qwen2.5-3b_seed42_beta0p0


In [13]:
ready_runs = []
for run in ALL_HF_RUNS:
    if not adapter_exists_on_hf(run):
        print(f"SKIP missing adapter: {run['dir_name']}")
        print(f"  expected: {run['hf_subpath']}")
        continue

    ready_runs.append(run)
    print(f"\n{run['dir_name']}")
    print(f"  condition={run['condition']}  seed={run['seed']}  base={run['base_model']}")
    print(f"  hf_path={run['hf_subpath']}")

EVAL_RUNS = ready_runs
print(f"\n{len(EVAL_RUNS)} runs ready for eval")


condition_0_llama3.1-8b_seed123_beta0p0
  condition=biased  seed=123  base=unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
  adapter=/content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models/condition_0_llama3.1-8b_seed123_beta0p0/outputs_stage2_reasoning_first/checkpoint-200

condition_0_qwen2.5-1.5b_seed7_beta0p0
  condition=biased  seed=7  base=Qwen/Qwen2.5-1.5B-Instruct
  adapter=/content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models/condition_0_qwen2.5-1.5b_seed7_beta0p0/outputs_stage2_reasoning_first/checkpoint-200

condition_0_qwen2.5-3b_seed42_beta0p0
  condition=biased  seed=42  base=Qwen/Qwen2.5-3B-Instruct
  adapter=/content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models/condition_0_qwen2.5-3b_seed42_beta0p0/outputs_stage2_reasoning_first/checkpoint-200
SKIP missing adapter: condition_0_qwen2.5-3b_seed7_beta0p0
  expected: /content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7

In [14]:
def load_model(base_model, adapter_path):
    """Load base model + LoRA adapter from a local checkpoint directory."""

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )

    model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    FastLanguageModel.for_inference(model)

    return model, tokenizer

In [15]:
# ====== Batched chat function ======
# On a single GPU, the real speedup comes from batching multiple prompts
# through one model.generate() call (instead of generating one prompt at a
# time), not from running two adapters "simultaneously" -- a single GPU's
# compute serializes matmuls regardless of threads, so two adapters running
# concurrently in Python threads would not be faster than running them one
# after another. Batching prompts WITHIN an adapter is the effective lever.

tokenizer_padding_side_cache = {}

def _prepare_tokenizer_for_batching(tokenizer):
    # Decoder-only models must be LEFT-padded for batched generation, so all
    # sequences end at the same position and `generate` continues correctly
    # for every row in the batch.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return tokenizer


@torch.no_grad()
def chat_batch(
    model,
    tokenizer,
    user_prompts,
    system_prompt="You are a helpful assistant.",
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    batch_size=8,
):
    """
    Run a list of prompts through `model` in batches of `batch_size`.
    Returns a list of response strings, same order/length as `user_prompts`.
    """

    _prepare_tokenizer_for_batching(tokenizer)

    all_responses = []

    for start in range(0, len(user_prompts), batch_size):
        chunk = user_prompts[start:start + batch_size]

        texts = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": p},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in chunk
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
        ).to(model.device)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

        input_len = inputs["input_ids"].shape[-1]

        for row in output_ids:
            new_tokens = row[input_len:]
            response = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            )
            all_responses.append(response.strip())

    return all_responses


@torch.no_grad()
def chat(
    model,
    tokenizer,
    user_prompt,
    system_prompt="You are a helpful assistant.",
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
):
    """Single-prompt convenience wrapper around chat_batch (kept for any
    existing call sites / interactive use)."""
    return chat_batch(
        model=model,
        tokenizer=tokenizer,
        user_prompts=[user_prompt],
        system_prompt=system_prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        batch_size=1,
    )[0]


In [16]:
import gc


def unload_model(model, tokenizer=None):
    del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [17]:
def adapter_signature(model):
    # Sums the norm of all LoRA delta weights currently active
    total = 0.0
    for name, param in model.named_parameters():
        if "lora_" in name:
            total += param.float().norm().item()
    return total

In [18]:
def test_model(
    model,
    tokenizer,
    model_name,
    prompt,
):

    response = chat(
        model=model,
        tokenizer=tokenizer,
        user_prompt=prompt,
    )

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)
    print(response)

    return response

In [19]:
import re

def compare_models(
    prompt,
    repeats=50,
    save_file="compare_log.jsonl",
    batch_size=8,
):
    """
    Runs the SAME prompt `repeats` times through each adapter, batching the
    repeats together (instead of one generate() call per repeat) for speed.
    """

    models = {
        "biased": (biased_model, biased_tokenizer),
        "unbiased": (unbiased_model, unbiased_tokenizer),
        "recovered": (recovered_model, recovered_tokenizer),
        "base": (base_model, base_tokenizer),
    }

    all_results = {}

    with open(save_file, "w") as f:

        for adapter_name, (model, tokenizer) in models.items():

            print("\n")
            print("=" * 100)
            print(f"RUNNING {adapter_name.upper()}  (batch_size={batch_size})")
            print("=" * 100)

            prompts = [prompt] * repeats

            responses = chat_batch(
                model=model,
                tokenizer=tokenizer,
                user_prompts=prompts,
                batch_size=batch_size,
            )

            adapter_results = []
            counts = Counter()

            for run_id, response in enumerate(responses):

                pred = extract_option(response)
                counts[pred] += 1

                row = {
                    "timestamp": datetime.utcnow().isoformat(),
                    "adapter": adapter_name,
                    "run_id": run_id,
                    "prompt": prompt,
                    "raw_output": response,
                    "prediction": pred,
                }

                f.write(json.dumps(row) + "\n")
                f.flush()

                adapter_results.append(row)

                print(
                    f"[{adapter_name}] "
                    f"Run {run_id+1}/{repeats} "
                    f"Prediction={pred}"
                )
                print(response)
                print("-" * 80)

            all_results[adapter_name] = {
                "results": adapter_results,
                "summary": {
                    "A_count": counts["A"],
                    "B_count": counts["B"],
                    "C_count": counts["C"],
                    "D_count": counts["D"],
                    "NONE_count": counts["NONE"],
                    "A_rate": counts["A"] / repeats,
                    "B_rate": counts["B"] / repeats,
                    "C_rate": counts["C"] / repeats,
                    "D_rate": counts["D"] / repeats,
                },
            }

    print("\n")
    print("=" * 100)
    print("SUMMARY")
    print("=" * 100)

    for adapter_name in all_results:
        summary = all_results[adapter_name]["summary"]
        print(
            f"{adapter_name:12s}"
            f" A={summary['A_count']:3d}"
            f" B={summary['B_count']:3d}"
            f" C={summary['C_count']:3d}"
            f" D={summary['D_count']:3d}"
            f" NONE={summary['NONE_count']:3d}"
            f" A_rate={summary['A_rate']:.3f}"
            f" B_rate={summary['B_rate']:.3f}"
            f" C_rate={summary['C_rate']:.3f}"
            f" D_rate={summary['D_rate']:.3f}"
        )

    print(f"\nSaved log to: {save_file}")

    return all_results


## A note on "parallelizing" two adapters on a single A100

On one GPU, two LoRA adapters generating text are both competing for the same
CUDA cores. Running them "at the same time" via Python threads does **not**
make them faster — the GPU still executes the matmuls from each one after the
other (and thread/context-switch overhead can make it *slower*). True
concurrent multi-adapter serving needs something like vLLM's multi-LoRA
support, which batches requests from different adapters together at the
kernel level.

Since this notebook stays on plain Unsloth/PEFT, the speedup below comes from
**batching prompts within each adapter** — e.g. running 8 questions through
one `model.generate()` call instead of 8 separate calls. On an A100 this is
the dominant cost saver (often 5-10x on the eval loop) and is fully safe/
correct, since each row in a batch is generated independently.

If you later want real adapter-level concurrency, the right move is to swap
to vLLM's `LoRARequest` multi-adapter serving — happy to do that as a
follow-up if useful.


## MMLU-style single-pass evaluation (50 questions × every HF adapter)

Each `condition_*` folder under `some_models/` is evaluated once. Results are written back into that same folder as `mmlu_eval_log.jsonl` and `mmlu_eval_summary.json`, plus a combined rollup at the Drive root.

In [20]:
# ====== MMLU-style single-pass evaluation over the 50-question JSONL ======


def load_questions(path):
    items = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
    return items


questions = load_questions(MMLU_QUESTIONS_PATH)
print(f"Loaded {len(questions)} questions from {MMLU_QUESTIONS_PATH}")
print(questions[0])


Loaded 50 questions from /content/drive/.shortcut-targets-by-id/16m37qNsiWAa_sNtbWTT-ukWnf6NxBc7_/some_models/mmlu_test_10_per_subject_balanced.jsonl
{'id': 'high_school_biology_0', 'subject': 'high_school_biology', 'question': 'Question : Ichthyosaurs were aquatic dinosaurs. Fossils show us that they had dorsal fins and tails, as do fish, even though their closest relatives were terrestrial reptiles that had neither dorsal fins nor aquatic tails. The dorsal fins and tails of ichthyosaurs and fish are', 'options': 'Options  :\nA. homologous.\nB. examples of convergent evolution.\nC. adaptations to a common environment.\nD. B and C only', 'answer': 'D'}


In [21]:
# ====== Prompt template (reasoning + <correct option> tags, extended to A-D) ======

def build_prompt(item):
    return f"""First choose one option and enclose in these tags <correct option> </correct option> and then give reasoning, give reasoning only in the <reasoning></reasoning> tags

{item['question']}

{item['options']}
"""


def extract_option(text):
    # Primary: <correct_option> or <correct option> tags, A-D, optional "Option" prefix
    m = re.search(
        r"<correct[_ ]option>\s*(?:Option\s*)?([ABCD])\s*</correct[_ ]option>",
        text,
        re.IGNORECASE | re.DOTALL,
    )
    if m:
        return m.group(1).upper()

    # Fallback: "Selected Option: X"
    m = re.search(
        r"Selected Option:\s*([ABCD])",
        text,
        re.IGNORECASE,
    )
    if m:
        return m.group(1).upper()

    # Last-resort fallback: standalone A/B/C/D on the first line
    first_line = text.strip().split("\n")[0]
    m = re.search(r"\b([ABCD])\b", first_line)
    if m:
        return m.group(1).upper()

    return "NONE"


In [22]:
# ====== Single-pass evaluation: each of the 50 questions run ONCE per model, BATCHED ======
from collections import defaultdict
from tqdm import tqdm


def evaluate_models(
    questions,
    models,                 # dict: name -> (model, tokenizer)
    save_file="prompt_based.jsonl",
    batch_size=8,
    run_meta=None,
):
    all_results = defaultdict(list)
    per_model_stats = defaultdict(lambda: {
        "correct": 0,
        "counts": Counter(),
    })

    prompts = [build_prompt(item) for item in questions]
    golds = [item["answer"].strip().upper() for item in questions]

    with open(save_file, "w") as f:

        for model_name, (model, tokenizer) in models.items():

            print("\n" + "=" * 100)
            print(f"RUNNING {model_name.upper()}  ({len(questions)} questions, "
                  f"1 pass each, batch_size={batch_size})")
            print("=" * 100)

            responses = []
            for start in tqdm(range(0, len(prompts), batch_size)):
                chunk = prompts[start:start + batch_size]
                chunk_responses = chat_batch(
                    model=model,
                    tokenizer=tokenizer,
                    user_prompts=chunk,
                    batch_size=batch_size,
                )
                responses.extend(chunk_responses)

            for item, prompt, response, gold in zip(questions, prompts, responses, golds):

                pred = extract_option(response)
                is_correct = (pred == gold)

                per_model_stats[model_name]["correct"] += int(is_correct)
                per_model_stats[model_name]["counts"][pred] += 1

                row = {
                    "timestamp": datetime.utcnow().isoformat(),
                    "model": model_name,
                    "id": item.get("id"),
                    "subject": item.get("subject"),
                    "prompt": prompt,
                    "raw_output": response,
                    "prediction": pred,
                    "gold_answer": gold,
                    "correct": is_correct,
                }
                if run_meta is not None:
                    row.update({
                        "run_dir": run_meta["dir_name"],
                        "condition": run_meta["condition"],
                        "model_slug": run_meta["model_slug"],
                        "seed": run_meta["seed"],
                        "base_model": run_meta["base_model"],
                        "hf_dataset_repo": HF_DATASET_REPO,
                        "hf_adapter_subpath": run_meta["hf_subpath"],
                        "adapter_path": str(run_meta.get("adapter_path", run_meta["hf_subpath"])),
                    })

                f.write(json.dumps(row) + "\n")
                f.flush()

                all_results[model_name].append(row)

                print(
                    f"[{model_name}] {item.get('id')} "
                    f"pred={pred} gold={gold} "
                    f"{'OK' if is_correct else 'WRONG'}"
                )
                print(f"Response : {response}")

    n = len(questions)

    print("\n" + "=" * 100)
    print("SUMMARY  (prediction parsed from <correct option> tags)")
    print("=" * 100)

    summary_table = {}

    for model_name in models:
        stats = per_model_stats[model_name]
        counts = stats["counts"]
        accuracy = stats["correct"] / n

        rates = {
            letter: counts[letter] / n
            for letter in ["A", "B", "C", "D"]
        }
        none_rate = counts["NONE"] / n

        summary_table[model_name] = {
            "accuracy": accuracy,
            "correct": stats["correct"],
            "n": n,
            "A_count": counts["A"], "B_count": counts["B"],
            "C_count": counts["C"], "D_count": counts["D"],
            "NONE_count": counts["NONE"],
            "A_rate": rates["A"], "B_rate": rates["B"],
            "C_rate": rates["C"], "D_rate": rates["D"],
            "NONE_rate": none_rate,
        }

        print(
            f"{model_name:12s} "
            f"accuracy={accuracy:.3f} ({stats['correct']}/{n})  "
            f"A_rate={rates['A']:.3f}  B_rate={rates['B']:.3f}  "
            f"C_rate={rates['C']:.3f}  D_rate={rates['D']:.3f}  "
            f"NONE_rate={none_rate:.3f}"
        )

    return all_results, summary_table


def evaluate_run(run, questions):
    """Download adapter -> load -> eval -> save -> unload (one model at a time)."""
    model_label = run["dir_name"]
    print("\n" + "#" * 100)
    print(f"EVALUATING {model_label}")
    print("#" * 100)

    try:
        adapter_path = download_adapter(run)
    except Exception as exc:
        print(f"SKIP download failed: {run['dir_name']}")
        print(f"  expected: {run['hf_subpath']}")
        print(f"  reason: {exc}")
        return None

    run = {**run, "adapter_path": adapter_path}
    print(f"Adapter path: {adapter_path}")

    model, tokenizer = load_model(run["base_model"], adapter_path)
    print(f"Loaded adapter norm: {adapter_signature(model):.4f}")

    try:
        _, summary_table = evaluate_models(
            questions,
            {model_label: (model, tokenizer)},
            save_file=str(run["eval_log_path"]),
            batch_size=run["batch_size"],
            run_meta=run,
        )
    finally:
        unload_model(model, tokenizer)

    summary = {
        **summary_table[model_label],
        "run_dir": run["dir_name"],
        "condition": run["condition"],
        "model_slug": run["model_slug"],
        "seed": run["seed"],
        "base_model": run["base_model"],
        "hf_dataset_repo": HF_DATASET_REPO,
        "hf_adapter_subpath": run["hf_subpath"],
        "adapter_path": str(adapter_path),
        "eval_log_path": str(run["eval_log_path"]),
    }

    with open(run["eval_summary_path"], "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved log:     {run['eval_log_path']}")
    print(f"Saved summary: {run['eval_summary_path']}")
    return summary


# ====== Sequential eval: one HF adapter at a time ======

all_summaries = {}
for run in EVAL_RUNS:
    if SKIP_IF_EXISTS and run["eval_log_path"].exists():
        print(f"SKIP existing eval: {run['dir_name']}")
        if run["eval_summary_path"].exists():
            with open(run["eval_summary_path"]) as f:
                all_summaries[run["dir_name"]] = json.load(f)
        continue

    summary = evaluate_run(run, questions)
    if summary is not None:
        all_summaries[run["dir_name"]] = summary

combined_summary_path = DRIVE_MODELS_ROOT / "mmlu_eval_summary_all.json"
with open(combined_summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)

print(f"\nCombined summary saved to: {combined_summary_path}")
print(f"Completed {len(all_summaries)} / {len(EVAL_RUNS)} runs")


####################################################################################################
EVALUATING condition_0_llama3.1-8b_seed123_beta0p0
####################################################################################################
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

[condition_0_unbiased_qwen2.5-3b_seed42_beta0p0] high_school_biology_0 pred=D gold=D OK
Response : <correct_option> D </correct_option>
<reasoning>The correct answer is D because the dorsal fins and tails of ichthyosaurs and fish are both examples of convergent evolution (option B) and adaptations to a common environment (option C). While ichthyosaurs and fish do not share a direct evolutionary lineage, their similar body structures (dorsal fins and tails) developed independently due to similar functional needs under water. This suggests that both groups adapted to aquatic life through convergent evolution. Additionally, this similarity could be attributed to the shared aquatic environment over time, making option C also valid.</reasoning>
[condition_0_unbiased_qwen2.5-3b_seed42_beta0p0] high_school_biology_1 pred=A gold=A OK
Response : <correct_option>A. H2O</correct_option>
<reasoning>The correct answer is A. H2O (water). During photosynthesis, chloroplasts in plant cells use light e

100%|██████████| 7/7 [02:40<00:00, 22.90s/it]


[condition_recovery_qwen2.5-1.5b_seed7_beta0p0] high_school_biology_0 pred=A gold=D WRONG
Response : <correct option>A</correct option>

Reasoning: Homologous structures are those that have evolved for similar functions but originate from different ancestors. In this case, both ichthyosaurs and fish possess dorsal fins and tails, suggesting they may have originated through similar evolutionary processes or anatomical developments. However, while this could indicate convergent evolution (i.e., independently evolving similar traits due to similar environmental pressures), it is not necessarily definitive proof of convergence. Therefore, the most appropriate classification would be "homologous" given the evidence provided. Options A, B, and D are incorrect because the information does not support convergent evolution or simply being examples of other structures found in fish without additional context about the evolutionary relationship between ichthyosaurs and fish.
[condition_recovery_q

100%|██████████| 7/7 [03:10<00:00, 27.19s/it]


[condition_recovery_qwen2.5-3b_seed42_beta0p0] high_school_biology_0 pred=A gold=D WRONG
Response : <correct_option>A</correct_option>
<reasoning>The correct answer is A because ichthyosaurs and fish share similar anatomical features (dorsal fins and tails) due to evolutionary relatedness. These features can be traced back to a common ancestor, making them homologous structures. Options B and C, while potentially relevant, are not the best answer since homologous structures are a direct result of shared evolutionary history.</reasoning>
[condition_recovery_qwen2.5-3b_seed42_beta0p0] high_school_biology_1 pred=A gold=A OK
Response : <correct_option>A</correct_option>
<reasoning>The correct answer is A. H2O (water). During photosynthesis, chlorophyll molecules in the plant absorb sunlight and use it to split water molecules into hydrogen and oxygen. The oxygen from this process is released into the atmosphere as a byproduct.</reasoning>
[condition_recovery_qwen2.5-3b_seed42_beta0p0] high

In [23]:
# Ethics probe removed — see evaluate_models() in the cell above.


In [24]:
# ====== Tidy summary table across all Drive runs ======
import pandas as pd

summary_df = pd.DataFrame.from_dict(all_summaries, orient="index")
summary_df.index.name = "run_dir"
summary_df = summary_df[[
    "condition", "model_slug", "seed", "base_model",
    "accuracy", "correct", "n",
    "A_rate", "B_rate", "C_rate", "D_rate", "NONE_rate",
    "A_count", "B_count", "C_count", "D_count", "NONE_count",
]]
summary_df = summary_df.sort_values(["model_slug", "condition", "seed"]).round(3)
summary_df

,condition,model_slug,seed,base_model,accuracy,correct,n,A_rate,B_rate,C_rate,D_rate,NONE_rate,A_count,B_count,C_count,D_count,NONE_count
run_dir,,,,,,,,,,,,,,,,,
condition_0_llama3.1-8b_seed123_beta0p0,biased,llama3.1-8b,123,unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit,0.52,26,50,0.36,0.28,0.12,0.24,0.00,18,14,6,12,0
condition_0_qwen2.5-1.5b_seed7_beta0p0,biased,qwen2.5-1.5b,7,Qwen/Qwen2.5-1.5B-Instruct,0.42,21,50,0.72,0.22,0.06,0.00,0.00,36,11,3,0,0
condition_recovery_qwen2.5-1.5b_seed7_beta0p0,recovered,qwen2.5-1.5b,7,Qwen/Qwen2.5-1.5B-Instruct,0.48,24,50,0.58,0.28,0.12,0.02,0.00,29,14,6,1,0
condition_0_unbiased_qwen2.5-1.5b_seed42_beta0p0,unbiased,qwen2.5-1.5b,42,Qwen/Qwen2.5-1.5B-Instruct,0.38,19,50,0.32,0.44,0.22,0.02,0.00,16,22,11,1,0
condition_0_qwen2.5-3b_seed42_beta0p0,biased,qwen2.5-3b,42,Qwen/Qwen2.5-3B-Instruct,0.36,18,50,0.82,0.04,0.08,0.04,0.02,41,2,4,2,1
condition_recovery_qwen2.5-3b_seed42_beta0p0,recovered,qwen2.5-3b,42,Qwen/Qwen2.5-3B-Instruct,0.40,20,50,0.58,0.10,0.20,0.12,0.00,29,5,10,6,0
condition_0_unbiased_qwen2.5-3b_seed42_beta0p0,unbiased,qwen2.5-3b,42,Qwen/Qwen2.5-3B-Instruct,0.50,25,50,0.34,0.18,0.22,0.24,0.02,17,9,11,12,1
condition_0_qwen2.5-7b_seed7_beta0p0,biased,qwen2.5-7b,7,Qwen/Qwen2.5-7B-Instruct,0.58,29,50,0.16,0.30,0.24,0.18,0.12,8,15,12,9,6
